## Retrieval Optimisation Techniques

### 1) Top-K Tradeoff

In [1]:
# We pick the most relevant documents/chunks

In [2]:
# Case 1

top_k = 1 # very low

# Pros 
#     High Precision
#     Fast processing as only one chunk
#     Cost effectvie
# Cons 
#     May miss out required info (context)

In [3]:
# Case 2

top_k = 10 # very large

# Pros 
#     More infor (context)
# Cons 
#     Low Precision
#     High Cost
#     May contain some irrelevant info
#     Slow Processings

In [4]:
# Case 3

top_k = 5 # medium

#### 1) Debugging Top-K

In [5]:
import os
import chromadb
from pathlib import Path
from openai import OpenAI

In [6]:
def load_documents(data_folder):
    documents= []
    doc_ids = []

    for file in os.listdir(data_folder):
        file_path = os.path.join(data_folder, file)

        with open(file_path, 'r') as f:
            text = f.read()
        
        documents.append(text)
        doc_ids.append(file.removesuffix('.txt'))

    return [documents, doc_ids]

In [7]:
def get_embeddings_from_llm(llm_client, text):
    response = llm_client.embeddings.create(
        model='text-embedding-3-small',
        input=text
    )

    return response.data[0].embedding

In [8]:
def store_embeddings(llm_client, documents):
    embeddings = []
    for doc in documents:
        emb = get_embeddings_from_llm(llm_client=llm_client, text=doc)
        embeddings.append(emb)

    return embeddings

In [9]:
def create_collection(db_client):
    collection = db_client.create_collection(name='company_policy_info_topk')
    return collection

In [10]:
def add_documents_to_collection(collection, documents, embeddings, ids):
    collection.add(
        documents=documents,
        embeddings=embeddings,
        ids=ids
    )

In [11]:
def get_context(collection, llm_client, query, top_k):
    query_emb = get_embeddings_from_llm(llm_client=llm_client, text=query)

    db_results = collection.query(
        query_embeddings = query_emb,
        n_results=top_k
    )

    docs = db_results['documents'][0]
    distances = db_results['distances'][0]

    for i in range(len(docs)):
        print(f'Rank: {i+1}')
        print(f'Distance: {distances[i]}')
        print(docs[i])
        print("---------------------------\n")

    return '\n'.join(docs)

In [12]:
def create_prompt(context, query):
    if not context.strip():
        prompt=f'''
        Answer the query based upon the given context only. Do not use your own knowledge base.
        No relevant context is found.
        QUERY:
        {query}
        '''
        return prompt
 
    prompt=f'''Answer the query based upon the given context only. Do not use your own knowledge base.
    CONTEXT:
    {context}
    QUERY:
    {query}
        '''

    return prompt

In [13]:
def generator(llm_client, prompt):
    response = llm_client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}]
    )

    return response.choices[0].message.content

In [14]:
# Create OpenAI & ChromaDB clients

openai_client = OpenAI(api_key=os.getenv('OPENAI_SECRET_KEY'))
db_client = chromadb.PersistentClient(path='./topk_chroma_db')

In [15]:
documents, doc_ids = load_documents(data_folder='data')
documents, doc_ids

(['AcmeTech Solutions Code of Conduct\n\nEmployees must maintain professional behavior at all times.\n\nHarassment, discrimination, or unethical conduct will not be tolerated.',
  'AcmeTech Solutions IT Security Policy\n\nEmployees must not share their passwords with anyone.\n\nAll systems must be locked when unattended.\n\nSensitive data must be stored securely and encrypted where appropriate.',
  'AcmeTech Solutions Leave Policy\n\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\n\nLeave requests must be submitted at least two weeks in advance.\n\nUnused leave may not be carried over to the next year unless approved.',
  'AcmeTech Solutions Remote Work Policy\n\nEmployees are allowed to work remotely up to 2 days per week.\n\nRemote work must be approved by the employeeâ€™s manager.\n\nEmployees must ensure a secure and productive work environment while working remotely.'],
 ['code_of_conduct',
  'it_security_policy',
  'leave_policy',
  'remot

In [16]:
embeddings = store_embeddings(llm_client=openai_client, documents=documents)

In [20]:
# db_client.delete_collection('company_policy_info_topk')

In [21]:
policy_collection = create_collection(db_client=db_client)

In [22]:
add_documents_to_collection(collection=policy_collection, 
                            documents=documents,
                            embeddings=embeddings, 
                            ids=doc_ids)

In [23]:
query = 'Can employees work remotely?'

topks = [1,3,5]

for k in topks:
    print(f"\n\n*********TOPK-{k}**********")
    context = get_context(collection=policy_collection, llm_client=openai_client, query=query, top_k=k)

    prompt = create_prompt(context=context, query=query)
    print("-----PROMPT-----")
    print(prompt)

    result = generator(llm_client=openai_client, prompt=prompt)
    print("-----RESULT-----")
    print(result)



*********TOPK-1**********
Rank: 1
Distance: 0.6608433127403259
AcmeTech Solutions Remote Work Policy

Employees are allowed to work remotely up to 2 days per week.

Remote work must be approved by the employeeâ€™s manager.

Employees must ensure a secure and productive work environment while working remotely.
---------------------------

-----PROMPT-----
Answer the query based upon the given context only. Do not use your own knowledge base.
    CONTEXT:
    AcmeTech Solutions Remote Work Policy

Employees are allowed to work remotely up to 2 days per week.

Remote work must be approved by the employeeâ€™s manager.

Employees must ensure a secure and productive work environment while working remotely.
    QUERY:
    Can employees work remotely?
        
-----RESULT-----
Yes, employees are allowed to work remotely up to 2 days per week, with approval from their manager.


*********TOPK-3**********
Rank: 1
Distance: 0.6608433127403259
AcmeTech Solutions Remote Work Policy

Employees are

In [24]:
# We cannot determine the optimal value for top-k.

### 2) Similarity Threshold

In [25]:
# distance-L2
# Lower the distance, closer the vectors

# Similarity= 1- distance
# Higher the similarity, closer the vectors

# Cosine Similarity
# It may be between -1 to +1
# Embedding models generally give the vectors where cosine similarity is from 0 to 1

In [26]:
# Score
# 1 - Most relevant (Almost same thing)
# 0.8 - Strongly similar
# 0.5 - Moderately similar
# 0.3 - Less similarity
# 0 - No similarity

# similarity >= 0.5 -> keep the chunk/document
# similarity < 0.5 -> discard

In [27]:
query = 'What is the capital of India?'

get_context(collection=policy_collection, llm_client=openai_client, query=query, top_k=1)

Rank: 1
Distance: 1.7765568494796753
AcmeTech Solutions IT Security Policy

Employees must not share their passwords with anyone.

All systems must be locked when unattended.

Sensitive data must be stored securely and encrypted where appropriate.
---------------------------



'AcmeTech Solutions IT Security Policy\n\nEmployees must not share their passwords with anyone.\n\nAll systems must be locked when unattended.\n\nSensitive data must be stored securely and encrypted where appropriate.'

In [28]:
def get_context_with_threshold(collection, llm_client, query, top_k, threshold=0.5):
    query_emb = get_embeddings_from_llm(llm_client=llm_client, text=query)

    db_results = collection.query(
        query_embeddings = query_emb,
        n_results=top_k,
    )

    docs = db_results['documents'][0]
    distances = db_results['distances'][0]

    filtered_docs = []
    print('Retrieval Debug Info...\n')

    for i, (doc, distance) in enumerate(zip(docs, distances)):
        similarity = 1 - distance
        print('Rank: ', i+1)
        print('Distance: ', distance)
        print('Similarity: ', similarity)

        if similarity>=threshold:
            filtered_docs.append(doc)
            print('Status: Accepted')
        else:
            print('Status: Rejected')

    return '\n'.join(filtered_docs)

In [29]:
query = 'What is the capital of India?'

get_context_with_threshold(collection=policy_collection, llm_client=openai_client, query=query, top_k=1)

Retrieval Debug Info...

Rank:  1
Distance:  1.7765568494796753
Similarity:  -0.7765568494796753
Status: Rejected


''

In [32]:
# db_client.delete_collection('threshold_test')

In [33]:
collection = db_client.create_collection(name='threshold_test',
                                         metadata={'hnsw:space': 'cosine'})
# chromadb will use cosine similarity instead of euclidian distance.
# hnsw is the algorithm.

In [34]:
add_documents_to_collection(collection=collection, 
                            documents=documents,
                            embeddings=embeddings, 
                            ids=doc_ids)

In [35]:
query = 'Can an employee work remotely'

get_context_with_threshold(collection=collection, llm_client=openai_client, query=query, top_k=5)

Retrieval Debug Info...

Rank:  1
Distance:  0.3478114604949951
Similarity:  0.6521885395050049
Status: Accepted
Rank:  2
Distance:  0.6542878150939941
Similarity:  0.34571218490600586
Status: Rejected
Rank:  3
Distance:  0.6607232093811035
Similarity:  0.3392767906188965
Status: Rejected
Rank:  4
Distance:  0.6942541599273682
Similarity:  0.30574584007263184
Status: Rejected


'AcmeTech Solutions Remote Work Policy\n\nEmployees are allowed to work remotely up to 2 days per week.\n\nRemote work must be approved by the employeeâ€™s manager.\n\nEmployees must ensure a secure and productive work environment while working remotely.'

In [36]:
query = 'What is the capital of India?'

get_context_with_threshold(collection=collection, llm_client=openai_client, query=query, top_k=5)

Retrieval Debug Info...

Rank:  1
Distance:  0.8882783651351929
Similarity:  0.11172163486480713
Status: Rejected
Rank:  2
Distance:  0.8935425877571106
Similarity:  0.1064574122428894
Status: Rejected
Rank:  3
Distance:  0.9237940907478333
Similarity:  0.07620590925216675
Status: Rejected
Rank:  4
Distance:  0.931885302066803
Similarity:  0.06811469793319702
Status: Rejected


''

In [37]:
query = 'Can an employee work remotely'

context = get_context_with_threshold(collection=collection, llm_client=openai_client, query=query, top_k=5, threshold=0.33)

Retrieval Debug Info...

Rank:  1
Distance:  0.3478114604949951
Similarity:  0.6521885395050049
Status: Accepted
Rank:  2
Distance:  0.6542878150939941
Similarity:  0.34571218490600586
Status: Accepted
Rank:  3
Distance:  0.6607232093811035
Similarity:  0.3392767906188965
Status: Accepted
Rank:  4
Distance:  0.6942541599273682
Similarity:  0.30574584007263184
Status: Rejected


In [38]:
prompt = create_prompt(context=context, query=query)
print("-----PROMPT-----")
print(prompt)

result = generator(llm_client=openai_client, prompt=prompt)
print("-----RESULT-----")
print(result)

-----PROMPT-----
Answer the query based upon the given context only. Do not use your own knowledge base.
    CONTEXT:
    AcmeTech Solutions Remote Work Policy

Employees are allowed to work remotely up to 2 days per week.

Remote work must be approved by the employeeâ€™s manager.

Employees must ensure a secure and productive work environment while working remotely.
AcmeTech Solutions Leave Policy

All full-time employees are entitled to 20 days of paid annual leave per calendar year.

Leave requests must be submitted at least two weeks in advance.

Unused leave may not be carried over to the next year unless approved.
AcmeTech Solutions IT Security Policy

Employees must not share their passwords with anyone.

All systems must be locked when unattended.

Sensitive data must be stored securely and encrypted where appropriate.
    QUERY:
    Can an employee work remotely
        
-----RESULT-----
Yes, an employee can work remotely, but it must be approved by the employee's manager, and

## Chunking

### Recursive chunking with overlapping

In [39]:
import re

In [40]:
# In Recursive Chunking, we split on pages/paragraphs/sentences/words

In [41]:
def split_sentences(text):
    return re.split(r'(?<=[.?!;])\s+',text)

In [71]:
split_sentences("Hello! How are you doing? I hope you are good.")

['Hello!', 'How are you doing?', 'I hope you are good.']

In [43]:
t = 'How are your doing?'
t[-10:]

'our doing?'

In [88]:
def get_chunks_recursively(text, chunk_size=120, overlap=15):
    sentences = split_sentences(text)

    chunks = []
    current_chunk = ''

    for sentence in sentences:
        if len(current_chunk) + len(sentence) <= chunk_size:
            # print("if--------")
            current_chunk += sentence + " "
        else:
            # print("else--------")
            chunks.append(current_chunk.strip())
            
            # handle overlapping
            overlap_text = current_chunk[-overlap:]
            current_chunk = overlap_text + sentence + " "

    if current_chunk.strip():
        chunks.append(current_chunk.strip())
        
    return chunks

### Chunk the documents

In [46]:
data_folder = "data"

In [100]:
all_chunks = []
metadata = []
ids = []

for file in os.listdir(data_folder):
    path=os.path.join(data_folder, file)
    with open(path) as f:
        text = f.read()
    chunks = get_chunks_recursively(text)

    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)

        metadata.append({'source': file,
                         'chunk_number': i})
        
        ids.append(f'{file}_chunk_{i}')

In [101]:
all_chunks

['AcmeTech Solutions Code of Conduct\n\nEmployees must maintain professional behavior at all times.',
 'at all times. Harassment, discrimination, or unethical conduct will not be tolerated.',
 'AcmeTech Solutions IT Security Policy\n\nEmployees must not share their passwords with anyone.',
 's with anyone. All systems must be locked when unattended.',
 'en unattended. Sensitive data must be stored securely and encrypted where appropriate.',
 'AcmeTech Solutions Leave Policy\n\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.',
 'calendar year. Leave requests must be submitted at least two weeks in advance.',
 'ks in advance. Unused leave may not be carried over to the next year unless approved.',
 'AcmeTech Solutions Remote Work Policy\n\nEmployees are allowed to work remotely up to 2 days per week.',
 "days per week. Remote work must be approved by the employee's manager.",
 "yee's manager. Employees must ensure a secure and productive work envir

In [104]:
metadata

[{'source': 'code_of_conduct.txt', 'chunk_number': 0},
 {'source': 'code_of_conduct.txt', 'chunk_number': 1},
 {'source': 'it_security_policy.txt', 'chunk_number': 0},
 {'source': 'it_security_policy.txt', 'chunk_number': 1},
 {'source': 'it_security_policy.txt', 'chunk_number': 2},
 {'source': 'leave_policy.txt', 'chunk_number': 0},
 {'source': 'leave_policy.txt', 'chunk_number': 1},
 {'source': 'leave_policy.txt', 'chunk_number': 2},
 {'source': 'remote_work_policy.txt', 'chunk_number': 0},
 {'source': 'remote_work_policy.txt', 'chunk_number': 1},
 {'source': 'remote_work_policy.txt', 'chunk_number': 2}]

In [94]:
ids

['code_of_conduct.txt_chunk_0',
 'code_of_conduct.txt_chunk_1',
 'it_security_policy.txt_chunk_0',
 'it_security_policy.txt_chunk_1',
 'it_security_policy.txt_chunk_2',
 'leave_policy.txt_chunk_0',
 'leave_policy.txt_chunk_1',
 'leave_policy.txt_chunk_2',
 'remote_work_policy.txt_chunk_0',
 'remote_work_policy.txt_chunk_1',
 'remote_work_policy.txt_chunk_2']

In [58]:
split_sentences(text)

['AcmeTech Solutions Code of Conduct\n\nEmployees must maintain professional behavior at all times.',
 'Harassment, discrimination, or unethical conduct will not be tolerated.']

In [95]:
get_chunks_recursively(text)

['AcmeTech Solutions Remote Work Policy\n\nEmployees are allowed to work remotely up to 2 days per week.',
 "days per week. Remote work must be approved by the employee's manager.",
 "yee's manager. Employees must ensure a secure and productive work environment while working remotely."]

### Add the chunks to the collection

In [ ]:
chunked_docs_collection = db_client.create_collection(name='chunked_docs', 
                                                      metadata={'hnsw:space': 'cosine'})

In [97]:
for chunk, metadata, id in zip(all_chunks, metadata, ids):
    emb = get_embeddings_from_llm(llm_client=openai_client, text=chunk)

    chunked_docs_collection.add(
        documents=[chunk],
        embeddings=[emb],
        metadatas=[metadata],
        ids=[id]
    )

### Verify Chunks

In [105]:
results = chunked_docs_collection.get()
results

{'ids': ['code_of_conduct.txt_chunk_0',
  'code_of_conduct.txt_chunk_1',
  'it_security_policy.txt_chunk_0',
  'it_security_policy.txt_chunk_1',
  'it_security_policy.txt_chunk_2',
  'leave_policy.txt_chunk_0',
  'leave_policy.txt_chunk_1',
  'leave_policy.txt_chunk_2',
  'remote_work_policy.txt_chunk_0',
  'remote_work_policy.txt_chunk_1',
  'remote_work_policy.txt_chunk_2'],
 'embeddings': None,
 'documents': ['AcmeTech Solutions Code of Conduct\n\nEmployees must maintain professional behavior at all times.',
  'at all times. Harassment, discrimination, or unethical conduct will not be tolerated.',
  'AcmeTech Solutions IT Security Policy\n\nEmployees must not share their passwords with anyone.',
  's with anyone. All systems must be locked when unattended.',
  'en unattended. Sensitive data must be stored securely and encrypted where appropriate.',
  'AcmeTech Solutions Leave Policy\n\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.',
  'calen

In [106]:
for i in range(min(7, len(results['documents']))):
    print(f"ID: {results['ids'][i]}")
    print(f"Metadata: {results['metadatas'][i]}")
    print(f"Chunk: {results['documents'][i]}")
    break

ID: code_of_conduct.txt_chunk_0
Metadata: {'chunk_number': 0, 'source': 'code_of_conduct.txt'}
Chunk: AcmeTech Solutions Code of Conduct

Employees must maintain professional behavior at all times.


## Retrieval

In [107]:
query = 'How many days an employee can work from home?'

query_emb = get_embeddings_from_llm(llm_client=openai_client, 
                                    text=query)

In [109]:
output = chunked_docs_collection.query(
    query_embeddings=[query_emb],
    n_results=3
)

In [112]:
docs = output['documents'][0]
distances = output['distances'][0]
metadatas = output['metadatas'][0]
ids = output['ids'][0]

In [113]:
for i in range(len(docs)):
    similarity = 1 - distances[i]

    print(f'Rank: {i+1}')
    print(f'Similarity: {similarity}')
    print(f'Source: {metadatas[i]}')
    print(f'Chunk: {docs[i]}')
    print("-"*100)

Rank: 1
Similarity: 0.6301044821739197
Source: {'source': 'remote_work_policy.txt', 'chunk_number': 1}
Chunk: days per week. Remote work must be approved by the employee's manager.
----------------------------------------------------------------------------------------------------
Rank: 2
Similarity: 0.6181962490081787
Source: {'chunk_number': 0, 'source': 'remote_work_policy.txt'}
Chunk: AcmeTech Solutions Remote Work Policy

Employees are allowed to work remotely up to 2 days per week.
----------------------------------------------------------------------------------------------------
Rank: 3
Similarity: 0.39496421813964844
Source: {'chunk_number': 0, 'source': 'leave_policy.txt'}
Chunk: AcmeTech Solutions Leave Policy

All full-time employees are entitled to 20 days of paid annual leave per calendar year.
----------------------------------------------------------------------------------------------------


In [119]:
context = get_context_with_threshold(collection=chunked_docs_collection, llm_client=openai_client, query=query, top_k=3)
# we get only the relevant chunks using threshold value

Retrieval Debug Info...

Rank:  1
Distance:  0.3698955178260803
Similarity:  0.6301044821739197
Status: Accepted
Rank:  2
Distance:  0.3818037509918213
Similarity:  0.6181962490081787
Status: Accepted
Rank:  3
Distance:  0.6050357818603516
Similarity:  0.39496421813964844
Status: Rejected


In [116]:
context

"days per week. Remote work must be approved by the employee's manager.\nAcmeTech Solutions Remote Work Policy\n\nEmployees are allowed to work remotely up to 2 days per week."

In [118]:
get_context(collection=chunked_docs_collection, llm_client=openai_client, query=query, top_k=3)
# we get all the chunks without any threshold

Rank: 1
Distance: 0.3698955178260803
days per week. Remote work must be approved by the employee's manager.
---------------------------

Rank: 2
Distance: 0.3818037509918213
AcmeTech Solutions Remote Work Policy

Employees are allowed to work remotely up to 2 days per week.
---------------------------

Rank: 3
Distance: 0.6050357818603516
AcmeTech Solutions Leave Policy

All full-time employees are entitled to 20 days of paid annual leave per calendar year.
---------------------------



"days per week. Remote work must be approved by the employee's manager.\nAcmeTech Solutions Remote Work Policy\n\nEmployees are allowed to work remotely up to 2 days per week.\nAcmeTech Solutions Leave Policy\n\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year."

## Metadata Filteration

In [120]:
metadata_f_collection = db_client.create_collection(name='metadata_filteration',
                                                    metadata={'hnsw:space': 'cosine'})

In [121]:
department_mapping = {
    'code_of_conduct.txt': "Admin",
    'it_security_policy.txt': 'IT',
    'leave_policy.txt': "HR",
    'remote_work_policy.txt': "HR"
}

In [122]:
all_chunks = []
metadata = []
ids = []

for file in os.listdir(data_folder):
    path=os.path.join(data_folder, file)
    with open(path) as f:
        text = f.read()
    chunks = get_chunks_recursively(text)

    for i, chunk in enumerate(chunks):
        emb = get_embeddings_from_llm(llm_client=openai_client, text=chunk)

        metadata = ({'source': file,
                     'department': department_mapping[file],
                     'document_type': 'policy',
                     'chunk_number': i})
        
        metadata_f_collection.add(
            documents=[chunk],
            embeddings=[emb],
            metadatas=[metadata],
            ids=[f'{file}_chunk_{i}']
        )

In [124]:
# verify the metadata
metadata_f_collection.get()

{'ids': ['code_of_conduct.txt_chunk_0',
  'code_of_conduct.txt_chunk_1',
  'it_security_policy.txt_chunk_0',
  'it_security_policy.txt_chunk_1',
  'it_security_policy.txt_chunk_2',
  'leave_policy.txt_chunk_0',
  'leave_policy.txt_chunk_1',
  'leave_policy.txt_chunk_2',
  'remote_work_policy.txt_chunk_0',
  'remote_work_policy.txt_chunk_1',
  'remote_work_policy.txt_chunk_2'],
 'embeddings': None,
 'documents': ['AcmeTech Solutions Code of Conduct\n\nEmployees must maintain professional behavior at all times.',
  'at all times. Harassment, discrimination, or unethical conduct will not be tolerated.',
  'AcmeTech Solutions IT Security Policy\n\nEmployees must not share their passwords with anyone.',
  's with anyone. All systems must be locked when unattended.',
  'en unattended. Sensitive data must be stored securely and encrypted where appropriate.',
  'AcmeTech Solutions Leave Policy\n\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.',
  'calen

### Filter by Metadata

In [127]:
query

'How many days an employee can work from home?'

In [141]:
db_results = metadata_f_collection.query(
    query_embeddings=[query_emb],
    n_results=3,
    where={'department': 'HR'}
)

db_results['documents'][0]

["days per week. Remote work must be approved by the employee's manager.",
 'AcmeTech Solutions Remote Work Policy\n\nEmployees are allowed to work remotely up to 2 days per week.',
 'AcmeTech Solutions Leave Policy\n\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.']

In [143]:
db_results = metadata_f_collection.query(
    query_embeddings=[query_emb],
    n_results=3,
    where={'department': 'IT'}
)

db_results['documents'][0]

['AcmeTech Solutions IT Security Policy\n\nEmployees must not share their passwords with anyone.',
 'en unattended. Sensitive data must be stored securely and encrypted where appropriate.',
 's with anyone. All systems must be locked when unattended.']

In [145]:
db_results = metadata_f_collection.query(
    query_embeddings=[query_emb],
    n_results=3,
    where={'document_type': 'policy'}
)

db_results['documents'][0]

["days per week. Remote work must be approved by the employee's manager.",
 'AcmeTech Solutions Remote Work Policy\n\nEmployees are allowed to work remotely up to 2 days per week.',
 'AcmeTech Solutions Leave Policy\n\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.']

In [146]:
docs = db_results['documents'][0]
distances = db_results['distances'][0]

filtered_docs = []
print('Retrieval Debug Info...\n')

for i, (doc, distance) in enumerate(zip(docs, distances)):
    similarity = 1 - distance
    print('Rank: ', i+1)
    print('Distance: ', distance)
    print('Similarity: ', similarity)

    if similarity>=0.5:
        filtered_docs.append(doc)
        print('Status: Accepted')
    else:
        print('Status: Rejected')

Retrieval Debug Info...

Rank:  1
Distance:  0.3698955178260803
Similarity:  0.6301044821739197
Status: Accepted
Rank:  2
Distance:  0.3818037509918213
Similarity:  0.6181962490081787
Status: Accepted
Rank:  3
Distance:  0.6050357818603516
Similarity:  0.39496421813964844
Status: Rejected


## Query Expansion (Multi Query Retrieval)

In [147]:
# query- What are the work from home rules?
# DB- Remote work policy

# sim=.44 but the threshold is .5
# It will cause retrieval failure

### Solution - Query Expansion

In [148]:
# We create multiple variations of a query (suppose-3)
# Create embedding of each variation
# Check similarity with each variation

In [150]:
# DB- Remote work policy

# Query-
#     What are the work from home rules? (User)

# Variations-
#     WFH policy
#     Offsite working policy
#     Hybrid working rules and regulations
#     Remote work is allowed or not?

#### Method 1 : Create Variations Manually

In [162]:
def expand_query(query):
    variations = [
        query,
        query.replace('work from home', 'remote work'),
        query.replace('policy', 'rules'),
        query.replace('rules', 'policy'),
        query.replace('work from home', 'WFH'),
    ]
    return list(set(variations))

In [163]:
expand_query('What is the work from home policy of the company')

['What is the WFH policy of the company',
 'What is the remote work policy of the company',
 'What is the work from home policy of the company',
 'What is the work from home rules of the company']

In [164]:
expand_query('explain the work from home rules?')

['explain the work from home policy?',
 'explain the work from home rules?',
 'explain the WFH rules?',
 'explain the remote work rules?']

#### Method 2 : Use LLM to create variations